In [2]:
"""Evaluate a models testing keyrank at N traces and how many traces required to achieve 99% accuracy"""

import data
import sys
import numpy as np
import torch
import json

import training

import keyrank_rs

import torch.distributed as dist

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

In [3]:
torch.use_deterministic_algorithms(False)

In [4]:
import torch.nn as nn

In [5]:
def get_best_epoch(model_name) -> int:
    with open(f"models/eval/{model_name}.txt", 'r') as f:
        lines = f.readlines()[1:] # skip "N traces" line
        lines = [float(line.strip()) for line in lines]
        best_epoch = np.array(lines).argmin()
    return best_epoch.item()

def metadata_best_epoch(model_name) -> int:
    with open(f"models/{model_name}/metadata.json") as f:
        metadata = json.load(f)
        val_scores = metadata["scores"][1]
        best_epoch = np.array(val_scores).argmin()
    return best_epoch.item()

def setup(rank, world_size):
    dist.init_process_group(
        backend="nccl",
        init_method="tcp://127.0.0.1:29500",
        rank=rank,
        world_size=world_size,
        device_id=None,
    )


device = torch.device("cuda")

testing_data = {}


IMPL = "fixslice"
ARCH = "zhang"

PREDICTION_TARGET = "2sbox"
TRAINING_BYTE = 1
TRACE_START = 400
TRACE_END = 1500
SEED = 777



In [6]:
if not PREDICTION_TARGET == "combo":
    model_name = f"{IMPL}-{PREDICTION_TARGET}-byte{TRAINING_BYTE}-{ARCH}-{TRACE_START}_{TRACE_END}-s{SEED}"

    if ARCH == "transnet":
        epoch = 199
    else:
        epoch = metadata_best_epoch(model_name)

    model_path = f"models/{model_name}/epoch{epoch}.pt"
    print(model_path)

    model = torch.load(model_path)
    if ARCH == "transnet":
        model = model.module
        
    model.eval()
else:
    models = []
    for target in ["sbox","sbox2"]:
        model_name = f"{IMPL}-{target}-byte{TRAINING_BYTE}-{ARCH}-{TRACE_START}_{TRACE_END}-s{SEED}"
        epoch = metadata_best_epoch(model_name)
        model_path = f"models/{model_name}/epoch{epoch}.pt"
        print(model_path)

        model = torch.load(model_path)
        model.eval()

        models.append(model)


testing_data["best_epoch"] = epoch

models/fixslice-2sbox-byte1-zhang-400_1500-s777/epoch40.pt


C:\Users\Ulrik\AppData\Local\Temp\ipykernel_21880\3213353238.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load(model_path)


In [7]:
from models import *


if ARCH == "transnet":
    try:
        setup(0, 1)
    except:
        pass


ATTACK_BYTE = 1
N_TRACES = 10



_,_,test_loader = data.get_dataloaders(
    200,
    PREDICTION_TARGET,
    ATTACK_BYTE,
    TRACE_START,
    TRACE_END,
    SEED,
)


testing_data["n_traces"] = N_TRACES






with torch.no_grad():
    if PREDICTION_TARGET == "key":
        testing_keyrank = training.mean_keyrank(model, test_loader, N_TRACES)
    elif PREDICTION_TARGET == "sbox":
        testing_keyrank = training.mean_sbox_rank(model, test_loader, N_TRACES)
    elif PREDICTION_TARGET == "sbox2":
        testing_keyrank = training.mean_sbox_rank(model, test_loader, N_TRACES, plaintext=1)
    elif PREDICTION_TARGET in ["2sbox", "2sbox*", "2sbox..."]:
        testing_keyrank = training.mean_2sbox_rank(model, test_loader, N_TRACES)
    elif PREDICTION_TARGET == "combo":
        testing_keyrank = None

testing_data["testing_keyrank"] = testing_keyrank
testing_keyrank + 1

tensor(1.4240, device='cuda:0', dtype=torch.float64)

In [19]:
def mean_2sbox_rank(model : nn.Module, sbox_test_loader, n_traces=500):
    
    total_rank = 0

    for traces, plaintexts, true_key in sbox_test_loader:

        traces : torch.Tensor = traces.to(device).squeeze()[0:n_traces]
        plaintexts : torch.Tensor = plaintexts.to(device)

        plaintexts = plaintexts.long().detach().cpu().numpy().squeeze()[0:n_traces]
        plaintexts : np.ndarray  = plaintexts.transpose((1, 0))


        sbox_scores = model(traces)
        # Handle both list and tensor with extra dimension
        if type(sbox_scores) == list:
            sbox_scores = torch.stack(sbox_scores).detach().cpu().numpy()
        else:
            sbox_scores = sbox_scores.detach().cpu().numpy()

        for scores, pt in zip(sbox_scores, plaintexts):

            numpy_keyscores = keyrank_rs.sbox_scores_to_keyscores_parallel(pt, scores)
            keyscores = torch.Tensor(numpy_keyscores)

            # logsum scores before calculating rank
            keyscores = keyscores.softmax(dim=1).log().sum(dim=0)

            indices = keyscores.argsort(dim=-1, descending=True)

            rank = np.where(indices == true_key)[0][0]

            total_rank += rank

    mean_rank = total_rank / (2.0 * len(sbox_test_loader))

    return mean_rank

In [22]:
with torch.no_grad():
    if PREDICTION_TARGET == "key":
        testing_keyrank = training.mean_keyrank(model, test_loader, N_TRACES)
    elif PREDICTION_TARGET == "sbox":
        testing_keyrank = training.mean_sbox_rank(model, test_loader, N_TRACES)
    elif PREDICTION_TARGET == "sbox2":
        testing_keyrank = training.mean_sbox_rank(model, test_loader, N_TRACES, plaintext=1)
    elif PREDICTION_TARGET in ["2sbox", "2sbox*", "2sbox..."]:
        testing_keyrank = mean_2sbox_rank(model, test_loader, N_TRACES)
    elif PREDICTION_TARGET == "combo":
        testing_keyrank = None

testing_keyrank

0.424

In [14]:
print(testing_keyrank)

None
